In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch, Rectangle
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box
from pymannkendall import original_test
from statsmodels.stats.multitest import multipletests
from pyproj import Transformer

Overview map of all observations

In [ ]:
df = pd.read_csv("../CWData_Switzerland.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
switzerland = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona")

counts = df.groupby(["latitude", "longitude"]).size().reset_index(name="count")

gdf_points = gpd.GeoDataFrame(
    counts,
    geometry=gpd.points_from_xy(counts["longitude"], counts["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:2056")

map_df_all = switzerland.to_crs("EPSG:2056")

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

# plot switzerland map
map_df_all.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# plot points
gdf_points.plot(
    ax=ax,
    color="teal",
    edgecolor="black",
    linewidth=0.3,
    alpha=0.7,
    markersize=np.log1p(gdf_points["count"]) * 40
)

legend_counts = [1, 10, 50, 100]
legend_handles = [
    plt.scatter([], [], s=np.log1p(c) * 40, color="teal", edgecolor="black", linewidth=0.3, alpha=0.7, label=str(c))
    for c in legend_counts
]
legend = ax.legend(
    handles=legend_handles,
    title="Observations",
    loc="upper left",
    frameon=True,
    labelspacing=1.5,
)
legend.get_title().set_fontsize(14)
for text in legend.get_texts():
    text.set_fontsize(11)

ax.set_title("All Observations in Switzerland", fontsize=16)
ax.set_axis_off()

plt.savefig("../Products/Switzerland/Switzerland_Overview.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_34720\2008410389.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Stream_Drainage_Basin, 34: Lake_Usage, 35: Lake_Usage_Nr, 36: Lake_Access, 37: Lake_Shore_State, 38: Lake_Swimming, 39: Lake_Transparency, 40: Lake_Color, 41: Lake_Odor, 42: Lake_Shore_Vegeta

KDE

In [ ]:
df = pd.read_csv("../CWData_Switzerland.csv")
df = df[(df["latitude"].between(-90, 90)) & (df["longitude"].between(-180, 180))]
world = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona").to_crs("EPSG:4326")

transformer = Transformer.from_crs("EPSG:4326", "EPSG:2056", always_xy=True)
x_m, y_m = transformer.transform(df["longitude"].values, df["latitude"].values)

# KDE
xy = np.vstack([x_m, y_m])
kde = gaussian_kde(xy, bw_method=0.05)  # change bw_method for more/less smooting

# grid
x_min, x_max = 2485000, 2834000
y_min, y_max = 1075000, 1296000

x_grid_m, y_grid_m = np.meshgrid(
    np.linspace(x_min, x_max, 800),
    np.linspace(y_min, y_max, 400)
)
grid_coords = np.vstack([x_grid_m.ravel(), y_grid_m.ravel()])

z = kde(grid_coords).reshape(x_grid_m.shape)
z_norm = z / z.max()
z_log = np.log1p(z_norm * 1000)
z_masked = np.where(z_norm < 0.001, np.nan, z_log) # do not show pixels without anything

transformer_back = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)
lon_grid, lat_grid = transformer_back.transform(x_grid_m, y_grid_m)

fig, ax = plt.subplots(figsize=(16, 8))
world.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.5, zorder=1)

# plot KDE
colors = ["#ffffff00", "#4393c3", "#2166ac", "#d6604d", "#b2182b"]
cmap = mcolors.LinearSegmentedColormap.from_list("cw_kde", colors)

vmin, vmax = np.nanmin(z_masked), np.nanpercentile(z_masked, 99) # clip highest percent

mesh = ax.pcolormesh(
    lon_grid, lat_grid, z_masked,
    cmap=cmap,
    shading="auto",
    zorder=2,
    alpha=0.85,
    vmax=vmax,
    vmin=vmin
)

world.plot(ax=ax, color="none", edgecolor="black", linewidth=0.5, zorder=3)

# colorbar
ticks = [vmin, vmin + (vmax-vmin)*0.25, vmin + (vmax-vmin)*0.5, vmin + (vmax-vmin)*0.75, vmax]
labels = ["Low", "", "Medium", "", "High"]

cbar = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.03, fraction=0.03)
cbar.set_label("Density (Log Scale)", fontsize=14)
cbar.set_ticks(ticks)
cbar.set_ticklabels(labels)
cbar.ax.tick_params(labelsize=11)

ax.set_title(" Density of Observations in Switzerland using Kernel Density Estimation (KDE)", fontsize=16)

ax.set_xlim(5.9, 10.6)
ax.set_ylim(45.8, 47.9)
ax.set_axis_off()

plt.savefig("../Products/Switzerland/KDE_Switzerland.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_34720\3773953437.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Stream_Drainage_Basin, 34: Lake_Usage, 35: Lake_Usage_Nr, 36: Lake_Access, 37: Lake_Shore_State, 38: Lake_Swimming, 39: Lake_Transparency, 40: Lake_Color, 41: Lake_Odor, 42: Lake_Shore_Vegeta

LISA

In [ ]:
df = pd.read_csv("../CWData_Switzerland.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
df["year"] = df["year_month"].dt.year

world = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona")

# create grid
gdf_points = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326"
).to_crs("EPSG:2056")
cell_size = 5000 # 5km cell size

xmin, ymin, xmax, ymax = world.total_bounds
x_bins = np.arange(xmin, xmax + cell_size, cell_size)
y_bins = np.arange(ymin, ymax + cell_size, cell_size)

grid = pd.MultiIndex.from_product(
    [y_bins, x_bins], names=["y_bin", "x_bin"]
).to_frame(index=False)

gdf_points["x_bin"] = (np.floor((gdf_points.geometry.x - xmin) / cell_size) * cell_size + xmin)
gdf_points["y_bin"] = (np.floor((gdf_points.geometry.y - ymin) / cell_size) * cell_size + ymin)

all_years = sorted(df["year"].unique())

# LISA colors
lisa_colors = {
    "HH": "red",  # High-High Hotspot
    "HL": "orange",  # High-Low Outlier
    "LH": "lightblue",  # Low-High Outlier
    "ns": "white"   # non-significant
}

# observations per grid cell
counts = (
    gdf_points
    .groupby(["y_bin", "x_bin"])
    .size()
    .reset_index(name="n_obs")
)

grid_active = counts[counts["n_obs"] > 0].copy() # ignore cells without any observations

grid_active["geometry"] = grid_active.apply(
    lambda row: box(row["x_bin"], row["y_bin"], row["x_bin"] + cell_size, row["y_bin"] + cell_size),
    axis=1
)
gdf_active = gpd.GeoDataFrame(grid_active, geometry="geometry", crs="EPSG:2056")

w = DistanceBand.from_dataframe(gdf_active, threshold=25000, silence_warnings=True)
isolated = [i for i, neighbors in w.neighbors.items() if len(neighbors) == 0]
gdf_active = gdf_active.drop(index=isolated).reset_index(drop=True)

w = DistanceBand.from_dataframe(gdf_active, threshold=25000, silence_warnings=True)
w.transform = "r"

y = gdf_active["n_obs"].values

# seed makes it reproducible
np.random.seed(1)

# calculate LISA
lisa = Moran_Local(y, w, permutations=999, seed=1)

# calculate Moran's I globally
moran = Moran(y, w, permutations=999)

# significance: p < 0.05
sig = lisa.p_sim < 0.1
quads = lisa.q  # 1=HH, 2=LH, 3=LL, 4=HL
quad_map = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}

gdf_active["lisa_cat"] = "ns"
gdf_active.loc[sig, "lisa_cat"] = [quad_map[q] for q in quads[sig]]

gdf_sig = gdf_active[gdf_active["lisa_cat"] != "ns"] # only plot significant cells


fig, ax = plt.subplots(1, 1, figsize=(16, 8))

world.plot(ax=ax, color="grey", edgecolor="black", linewidth=0.5, zorder=1)

for cat, color in lisa_colors.items():
    if cat == "ns":
        continue
    subset = gdf_sig[gdf_sig["lisa_cat"] == cat]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, zorder=2, alpha=0.8)

patches = [
    mpatches.Patch(facecolor=lisa_colors["HH"], edgecolor="black", linewidth=0.5, label="High-High (Hotspot)"),
    mpatches.Patch(facecolor=lisa_colors["HL"], edgecolor="black", linewidth=0.5, label="High-Low (Outlier)"),
    mpatches.Patch(facecolor=lisa_colors["LH"], edgecolor="black", linewidth=0.5, label="Low-High (Outlier)"),
]
legend = ax.legend(handles=patches, title="LISA Cluster (α = 0.1)", loc="lower left", frameon=True)
legend.get_title().set_fontsize(14)
for text in legend.get_texts():
    text.set_fontsize(11)

ax.set_title(f"Local Indicators of Spatial Autocorrelation (LISA) in Switzerland", fontsize=16)

# write Moran's I on map
ax.text(
    0.011, 0.93,
    f"Moran's I = {moran.I:.3f} (p = {moran.p_sim:.3f})",
    transform=ax.transAxes,
    fontsize=12,
    verticalalignment="bottom",
    bbox=dict(boxstyle="round", facecolor="white")
)

ax.set_axis_off()

plt.savefig(f"../Products/Switzerland/Switzerland_LISA.png", dpi=300, bbox_inches="tight")
plt.close()


C:\Users\yanni\AppData\Local\Temp\ipykernel_34720\1771079236.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Stream_Drainage_Basin, 34: Lake_Usage, 35: Lake_Usage_Nr, 36: Lake_Access, 37: Lake_Shore_State, 38: Lake_Swimming, 39: Lake_Transparency, 40: Lake_Color, 41: Lake_Odor, 42: Lake_Shore_Vegeta

Map with number of observations per 100km2, canton-wise

In [30]:
df = pd.read_csv("../CWData_Switzerland.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

# dataframe with country and the number of observations in it
obs_per_canton = (
    df
    .groupby("Region")
    .agg(
        total_observations = ("created_at_local","count"),
    )
    .reset_index()
)

pd.set_option("display.max_rows", None)

world = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona") # higher resolution map because of area accuracy
world = world.rename(columns={"NAME": "Region"})

CANTON_NAME_FIXES = {
    "St. Gallen": "Sankt Gallen",
    "Luzern": "Lucerne",
}
world["Region"] = world["Region"].replace(CANTON_NAME_FIXES)

map_df = world.merge(obs_per_canton, on="Region", how="left")
map_df["area_km2"] = map_df.geometry.area / 1e6

bins = [0,0.01,0.1,1,5,50,float("inf")]
labels = ["<0.01","0.01-0.1","0.11-1","1-5","5-50",">50"]
map_df["obs_per_10km2"] = map_df["total_observations"] / map_df["area_km2"] * 10
map_df["cat_obs_per_10km2"] = pd.cut(map_df["obs_per_10km2"], bins=bins, labels=labels)

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

cmap = get_cmap("Reds", len(labels))
legend_handles = [
    Patch(facecolor=cmap(i / (len(labels) - 1)), edgecolor="black", linewidth=0.5, label=labels[i])
    for i in range(len(labels))
]
legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))

map_df.plot(
    column="cat_obs_per_10km2",
    cmap="Reds",
    linewidth=0.5,
    edgecolor="black",
    missing_kwds={"color": "grey"},
    legend=False,
    categorical=True,
    ax=ax
)

ax.legend(handles=legend_handles, title="Obs/10km²", fontsize=12, title_fontsize=15, loc="upper left")
ax.set_title("Number of Observations per 10 km²", fontsize=18)
ax.set_axis_off()

plt.savefig(f"../Products/Switzerland/number_of_obs_per_10km2_per_Canton.png", dpi=300, bbox_inches="tight")
plt.close()
map_df.to_csv("../Products/CSVs/obs_per_10kms_Switzerland.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_34720\2664307149.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Stream_Drainage_Basin, 34: Lake_Usage, 35: Lake_Usage_Nr, 36: Lake_Access, 37: Lake_Shore_State, 38: Lake_Swimming, 39: Lake_Transparency, 40: Lake_Color, 41: Lake_Odor, 42: Lake_Shore_Vegeta

In [26]:
print(sorted(df["Region"].unique()))
print(sorted(world["Region"].unique()))

['Aargau', 'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'Basel-Landschaft', 'Basel-Stadt', 'Bern', 'Fribourg', 'Genève', 'Glarus', 'Graubünden', 'Jura', 'Lucerne', 'Neuchâtel', 'Nidwalden', 'Obwalden', 'Sankt Gallen', 'Schaffhausen', 'Schwyz', 'Solothurn', 'Thurgau', 'Ticino', 'Uri', 'Valais', 'Vaud', 'Zug', 'Zürich']
['Aargau', 'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'Basel-Landschaft', 'Basel-Stadt', 'Bern', 'Fribourg', 'Genève', 'Glarus', 'Graubünden', 'Jura', 'Luzern', 'Neuchâtel', 'Nidwalden', 'Obwalden', 'Schaffhausen', 'Schwyz', 'Solothurn', 'St. Gallen', 'Thurgau', 'Ticino', 'Uri', 'Valais', 'Vaud', 'Zug', 'Zürich']


Number of observations, number of users and average number of observations per user per canton

In [ ]:
df = pd.read_csv("../CWData_Switzerland.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

# data
canton_freq = df["Region"].value_counts()
users_per_canton = df.groupby("Region")["created_by"].nunique().sort_values(ascending=False)
obs_per_user_canton = (canton_freq / users_per_canton).dropna()
canton_order = canton_freq.sort_values(ascending=False).index

# obs per canton
canton_obs = canton_freq[canton_order]

# users per canton
canton_users = users_per_canton[canton_order]

# avg obs per user per canton
canton_opu = obs_per_user_canton[canton_order]


# plot
fig, axes = plt.subplots(3, 1, figsize=(16, 23), sharex=True)

canton_obs.plot(kind="bar", color="teal", ax=axes[0], logy=True, fontsize=13)
axes[0].set_title("a) Number of Observations per Canton", fontsize=21, fontweight="bold", loc="left")
axes[0].set_ylabel("Number of Observations", fontsize=18)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
axes[0].set_axisbelow(True)

canton_users.plot(kind="bar", color="teal", ax=axes[1], logy=True, fontsize=13)
axes[1].set_title("b) Number of Users per Canton", fontsize=21, fontweight="bold", loc="left")
axes[1].set_ylabel("Number of Users", fontsize=18)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].set_axisbelow(True)

canton_opu.plot(kind="bar", color="teal", ax=axes[2], fontsize=13)
axes[2].set_title("c) Average Number of Observations per User per Canton", fontsize=21, fontweight="bold", loc="left")
axes[2].set_ylabel("Average Number of Observations per User", fontsize=18)
axes[2].grid(axis="y", linestyle="--", alpha=0.5)
axes[2].set_axisbelow(True)
axes[2].set_xlabel("Canton", fontsize=18)
axes[2].tick_params(axis="x", rotation=90)

# alternating background
for i in range(0, len(canton_obs), 2):
    for ax in axes:
        ax.axvspan(i - 0.5, i + 0.5, color="grey", alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig("../Products/Switzerland/Switzerland_canton_stats_combined.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_34720\3864469316.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Stream_Drainage_Basin, 34: Lake_Usage, 35: Lake_Usage_Nr, 36: Lake_Access, 37: Lake_Shore_State, 38: Lake_Swimming, 39: Lake_Transparency, 40: Lake_Color, 41: Lake_Odor, 42: Lake_Shore_Vegeta

In [31]:
canton_users.sort_values()

Region
Appenzell Ausserrhoden       1
Jura                         3
Schaffhausen                 3
Neuchâtel                    5
Appenzell Innerrhoden        5
Genève                       7
Glarus                       8
Uri                          8
Nidwalden                    8
Basel-Stadt                 12
Fribourg                    15
Zug                         18
Vaud                        19
Basel-Landschaft            22
Ticino                      25
Valais                      30
Lucerne                     30
Obwalden                    36
Solothurn                   41
Schwyz                      44
Thurgau                     49
Graubünden                  62
Bern                       102
Sankt Gallen               148
Aargau                     257
Zürich                    1260
Name: created_by, dtype: int64

Total monthly observations in Switzerland

In [ ]:
df = pd.read_csv("../CWData_Switzerland.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
monthly = df.groupby("year_month").size().reset_index(name="n_obs")
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 9))
ax.scatter(monthly["year_month_dt"], monthly["n_obs"], color="teal", s=20, zorder=3)
ax.plot(monthly["year_month_dt"], monthly["n_obs"], color="teal", linewidth=0.8, alpha=0.4)

ax.fill_between(monthly["year_month_dt"], monthly["n_obs"], alpha=0.1, color="teal")
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.title("Total Monthly Observations in Switzerland", fontsize=20)
plt.xlabel("Time", fontsize=17)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

plt.savefig(f"../Products/Switzerland/Switzerland_Lineplot_monthly_Observations.png", dpi=300, bbox_inches="tight")
plt.close()

monthly

C:\Users\yanni\AppData\Local\Temp\ipykernel_34720\3487916489.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Stream_Drainage_Basin, 34: Lake_Usage, 35: Lake_Usage_Nr, 36: Lake_Access, 37: Lake_Shore_State, 38: Lake_Swimming, 39: Lake_Transparency, 40: Lake_Color, 41: Lake_Odor, 42: Lake_Shore_Vegeta

,year_month,n_obs,year_month_dt
0,2017-02,23,2017-02-01
1,2017-03,37,2017-03-01
2,2017-04,29,2017-04-01
3,2017-05,49,2017-05-01
4,2017-06,59,2017-06-01
...,...,...,...
108,2026-02,126,2026-02-01
109,2026-03,234,2026-03-01
110,2026-04,271,2026-04-01
111,2026-05,258,2026-05-01


In [9]:
monthly = df.groupby("year_month").size().reset_index(name="n_obs")
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

result = own.STL_decomposition(monthly, "n_obs", "Monthly Observations", "Observations")

Seasonal strength Monthly Observations: 0.669
Trend strength Monthly Observations:    0.868


In [10]:
trend = result.trend.dropna()
mk = original_test(trend.values)

print(f"Trend: {mk.trend}")
print(f"p-value: {mk.p:.3f}")
print(f"Sen's slope: {mk.slope:.3f} obs/month")
print(f"Tau: {mk.Tau:.3f}")

Trend: increasing
p-value: 0.002
Sen's slope: 0.977 obs/month
Tau: 0.201


In [11]:
# trend since 2019
trend_2019 = trend[trend.index >= "2019-01-01"]
mk_2019 = original_test(trend_2019.values)
print(f"Trend: {mk_2019.trend}")
print(f"p-value: {mk_2019.p:.3f}")
print(f"Sen's slope: {mk_2019.slope:.3f} obs/month")
print(f"Tau: {mk_2019.Tau:.3f}")

Trend: decreasing
p-value: 0.023
Sen's slope: -1.068 obs/month
Tau: -0.164


In [12]:
# trend since 2021
trend_2021 = trend[trend.index >= "2021-01-01"]
mk_2021 = original_test(trend_2021.values)
print(f"Trend: {mk_2021.trend}")
print(f"p-value: {mk_2021.p:.3f}")
print(f"Sen's slope: {mk_2021.slope:.3f} obs/month")
print(f"Tau: {mk_2021.Tau:.3f}")

Trend: decreasing
p-value: 0.000
Sen's slope: -5.345 obs/month
Tau: -0.712


Persistent spots Switzerland only

In [13]:
def persistent_mapper_Switzerland(filename):
    persistent = pd.read_csv(f"../Products/CSVs/{filename}.csv")
    persistent = persistent[persistent["Country"] == "Switzerland"]
    world = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona")

    category_colors = {
        "physical scale": "#1418fc",
        "plastic pollution": "#fbff2b",
        "soil moisture": "#82571b",
        "standing water type": "#dd6ef0",
        "stream type": "#fc1471",
        "temporary stream": "#8c14fc",
        "virtual scale": "#14c2fc"
    }

    gdf = gpd.GeoDataFrame(
        persistent,
        geometry=gpd.points_from_xy(persistent["longitude"], persistent["latitude"]),
        crs="EPSG:4326"
    ).to_crs("EPSG:2056")
    gdf = gdf.sort_values("n_users", ascending=False)

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    world.to_crs("EPSG:2056").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    for cat, color in category_colors.items():
        subset = gdf[gdf["Category"] == cat]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x,
            subset.geometry.y,
            s=np.log1p(subset["n_users"]) * 70,
            color=color,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.8,
            label=cat,
            zorder=2
        )

    # legend categories
    legend_cat_handles = [
        plt.scatter([], [], s=50, color=color, edgecolor="black", linewidth=0.3, alpha=0.8, label=cat.title())
        for cat, color in category_colors.items()
        if cat in gdf["Category"].unique()
    ]
    legend_cats = ax.legend(
        handles=legend_cat_handles,
        title="Category",
        loc="lower left",
        frameon=True,
        title_fontsize=15,
        fontsize=12
    )

    # legend user counts
    user_counts = [1, 5, 10, 20]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(u) * 70, color="lightgrey",
                    edgecolor="black", linewidth=0.3, alpha=0.8, label=str(u))
        for u in user_counts
    ]
    legend_users = ax.legend(
        handles=legend_handles,
        title="Number of Unique Users",
        title_fontsize=14,
        fontsize=11,
        loc="upper left",
        frameon=True,
    )
    ax.add_artist(legend_cats)  # show both legends

    if "14d" in filename:
        ax.set_title("Persistent Spots (Every 14 Days for ≥180 Days)", fontsize=16)
        outpath = "../Products/Switzerland/Switzerland_persistent_spots_map_14d.png"
    else:
        ax.set_title("Persistent Spots (Every 30 Days for ≥365 Days)", fontsize=16)
        outpath = "../Products/Switzerland/Switzerland_persistent_spots_map_30d.png"

    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()

In [14]:
for file in ["persistent_spots_30d", "persistent_spots_14d"]:
    persistent_mapper_Switzerland(file)

Stats on persistent spots

In [15]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
df1 = df1[df1["Country"] == "Switzerland"]
df2 = df2[df2["Country"] == "Switzerland"]

df1_cantons = df1.groupby("Region").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_cantons = df2.groupby("Region").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_cantons.merge(df2_cantons, on="Region", how="outer", suffixes=("_14d", "_30d")).fillna(0).astype({"n_spots_14d": int, "n_spots_30d": int}).sort_values("Region", ascending=True)

df_combined

,Region,n_spots_14d,n_spots_30d
0,Bern,13,7
1,Schwyz,1,1
2,Solothurn,1,1
3,Vaud,2,3
4,Zürich,38,42


In [16]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
df1 = df1[df1["Country"] == "Switzerland"]
df2 = df2[df2["Country"] == "Switzerland"]

df1_cantons = df1.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_cantons = df2.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_cantons.merge(df2_cantons, on="Category", how="outer", suffixes=("_14d", "_30d")).fillna(0).astype({"n_spots_14d": int, "n_spots_30d": int}).sort_values("n_spots_14d", ascending=False)

df_combined

,Category,n_spots_14d,n_spots_30d
3,temporary stream,32,39
4,virtual scale,21,12
0,physical scale,1,1
1,soil moisture,1,1
2,stream type,0,1


In [17]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
df1 = df1[df1["Country"] == "Switzerland"]
df2 = df2[df2["Country"] == "Switzerland"]
spots1 = df1[["latitude", "longitude"]].drop_duplicates()
spots2 = df2[["latitude", "longitude"]].drop_duplicates()
spot_counts1 = df1.groupby(["latitude", "longitude", "Category"]).size().reset_index(name="n_streaks")
spot_counts2 = df2.groupby(["latitude", "longitude", "Category"]).size().reset_index(name="n_streaks")

in_both = spots1.merge(spots2, on=["latitude", "longitude"], how="inner")

categories_per_spot1 = df1.groupby(["latitude", "longitude"])["Category"].nunique().reset_index(name="n_categories")
categories_per_spot2 = df2.groupby(["latitude", "longitude"])["Category"].nunique().reset_index(name="n_categories")
multi_category_spots1 = categories_per_spot1[categories_per_spot1["n_categories"] > 1]
multi_category_spots2 = categories_per_spot2[categories_per_spot2["n_categories"] > 1]

print(f"Number of identical spots: {len(in_both)}")
print(f"Set 14d number of total spots: {len(df1)}")
print(f"Set 30d number of total spots: {len(df2)}")
print(f"Set 14d total unique spots: {len(spots1)}")
print(f"Set 30d total unique spots: {len(spots2)}")
print(f"Set 14d spots that are also in set 30d: {len(in_both)} ({len(in_both)/len(spots1)*100:.1f}%)")
print(f"Set 30d spots that are also in set 14d: {len(in_both)} ({len(in_both)/len(spots2)*100:.1f}%)")
print(f"Set 14d: {len(multi_category_spots1)} spots with more than one category")
print(f"Set 30d: {len(multi_category_spots2)} spots with more than one category")

print(spot_counts1["n_streaks"].value_counts().sort_index())
print(spot_counts2["n_streaks"].value_counts().sort_index())

Number of identical spots: 43
Set 14d number of total spots: 55
Set 30d number of total spots: 54
Set 14d total unique spots: 43
Set 30d total unique spots: 48
Set 14d spots that are also in set 30d: 43 (100.0%)
Set 30d spots that are also in set 14d: 43 (89.6%)
Set 14d: 0 spots with more than one category
Set 30d: 1 spots with more than one category
n_streaks
1    36
2     3
3     3
4     1
Name: count, dtype: int64
n_streaks
1    44
2     5
Name: count, dtype: int64
